In [ ]:

import os
import pandas as pd
from google.cloud import bigquery
from google.cloud import aiplatform


In [ ]:

PROJECT_ID = "dazzling-seat-366014"
REGION = "us-central1"
DATASET_ID = "forecasting"
TABLE_NAME = "sales_daily"
BUCKET_URI = "gs://dazzling-seat-366014-vertex-pipelines"


In [ ]:

def get_bq_data(project_id, dataset_id, table_name):
    client = bigquery.Client(project=project_id)
    query = f'''
    SELECT unique_id, date as ds, target as y
    FROM `{project_id}.{dataset_id}.{table_name}`
    '''
    df = client.query(query).to_dataframe()
    df['ds'] = pd.to_datetime(df['ds'])
    return df


In [ ]:

def preprocess_and_split(df):
    df_monthly = df.groupby(['unique_id', pd.Grouper(key='ds', freq='MS')])['y'].sum().reset_index()
    counts = df_monthly.groupby('unique_id').size()
    train_ids = counts[counts >= 24].index
    return df_monthly[df_monthly['unique_id'].isin(train_ids)]


In [ ]:

from mlforecast import MLForecast
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

def run_ml_forecast(df):
    models = {
        'xgb': XGBRegressor(),
        'lgbm': LGBMRegressor(),
        'lr': LinearRegression()
    }
    mlf = MLForecast(models=models, freq='MS', lags=[1,12])
    mlf.fit(df)
    preds = mlf.predict(12)
    return preds


In [ ]:

df = get_bq_data(PROJECT_ID, DATASET_ID, TABLE_NAME)
df = preprocess_and_split(df)
preds = run_ml_forecast(df)

preds.to_csv("forecast.csv", index=False)


In [ ]:

import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--ExecutePreprocessor.timeout=3600",
        "--inplace",
        "main.ipynb",
    ],
    capture_output=True,
    text=True,
    cwd="/Users/user/Documents/Vertex-ai",
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"main.ipynb execution failed (exit {result.returncode})")
print("main.ipynb executed successfully.")


In [1]:

import os
import pandas as pd
from google.cloud import bigquery
from google.cloud import aiplatform


In [2]:

PROJECT_ID = "dazzling-seat-366014"
REGION = "us-central1"
DATASET_ID = "forecasting"
TABLE_NAME = "sales_daily"
BUCKET_URI = "gs://dazzling-seat-366014-vertex-pipelines"


In [3]:

def get_bq_data(project_id, dataset_id, table_name):
    client = bigquery.Client(project=project_id)
    query = f'''
    SELECT unique_id, date as ds, target as y
    FROM `{project_id}.{dataset_id}.{table_name}`
    '''
    df = client.query(query).to_dataframe()
    df['ds'] = pd.to_datetime(df['ds'])
    return df


In [4]:

def preprocess_and_split(df):
    df_monthly = df.groupby(['unique_id', pd.Grouper(key='ds', freq='MS')])['y'].sum().reset_index()
    counts = df_monthly.groupby('unique_id').size()
    train_ids = counts[counts >= 24].index
    return df_monthly[df_monthly['unique_id'].isin(train_ids)]


In [5]:

from mlforecast import MLForecast
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

def run_ml_forecast(df):
    models = {
        'xgb': XGBRegressor(),
        'lgbm': LGBMRegressor(),
        'lr': LinearRegression()
    }
    mlf = MLForecast(models=models, freq='MS', lags=[1,12])
    mlf.fit(df)
    preds = mlf.predict(12)
    return preds


In [6]:

df = get_bq_data(PROJECT_ID, DATASET_ID, TABLE_NAME)
df = preprocess_and_split(df)
preds = run_ml_forecast(df)

preds.to_csv("forecast.csv", index=False)


/Users/user/Documents/Vertex-ai/.venv-1/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000344 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 18018, number of used features: 2
[LightGBM] [Info] Start training from score 4627.656383


In [7]:

import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--ExecutePreprocessor.timeout=3600",
        "--inplace",
        "main.ipynb",
    ],
    capture_output=True,
    text=True,
    cwd="/Users/user/Documents/Vertex-ai",
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"main.ipynb execution failed (exit {result.returncode})")
print("main.ipynb executed successfully.")



usage: python -m jupyter [-h] [--json] [--debug]
                         [--version | --config-dir | --data-dir |
                         --runtime-dir | --paths | subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbconvert` not found.



RuntimeError: main.ipynb execution failed (exit 1)